---
### **Robustness Test**

강건성 검사
- 분위수 검증
- 5x5 Double Sort
- 민감도 분석
- 스프레드 회귀

---
## **0. 포트폴리오 계산**

##### **데이터 로드**

In [ ]:
import pandas as pd
import numpy as np

total_adj_close     = pd.read_csv('../../00_input/KOSPI_KOSDAQ_total_adj_close.csv', index_col='Date', parse_dates=True)
trading_value_60    = pd.read_csv('../../00_input/KOSPI_KOSDAQ_trading_value_60.csv', index_col='Date', parse_dates=True)
trading_value       = pd.read_csv('../../00_input/KOSPI_KOSDAQ_trading_value.csv', index_col='Date', parse_dates=True)
mkt_cap             = pd.read_csv('../../00_input/KOSPI_KOSDAQ_mkt_cap.csv', index_col='Date', parse_dates=True)
adj_close           = pd.read_csv('../../00_input/KOSPI_KOSDAQ_adj_close.csv', index_col='Date', parse_dates=True)
factor_df           = pd.read_csv("../../00_input/safety_v1.csv", index_col=0, parse_dates=True) 

##### **포트폴리오 함수 정의**

In [2]:
from joblib import Parallel, delayed
from tqdm import tqdm

def run_strategy(
        wins_threshold_temp, weight_method_temp, cost_temp,
        trading_threshold_temp, Amihud_threshold_temp, n_temp,
        monthly_rets, illiq,
        sort_mode  = 'single',  # 'single', 'double', 'spread'
        c_temp     = None,      # Double Sort 시가총액 분위수
        spread_leg = None       # Spread 회귀 ('long' or 'short')
):
    
    high_cost, low_cost = cost_temp
    NAV = initial_NAV

    # 포트폴리오 이름 생성
    if sort_mode == 'single':
        portfolio_name = f"{weight_method_temp}_R{wins_threshold_temp}_H{int(high_cost*10000)}L{int(low_cost*10000)}_T{trading_threshold_temp}_A{Amihud_threshold_temp}_Q({n_temp+1}/{q_cut})"
    elif sort_mode == 'double': # For Double Sorting
        portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({n_temp+1}/{q_cut})"
    elif sort_mode == 'spread': # For Spread Regression
        portfolio_name = f"C({c_temp+1}/{cap_cut})_Q({spread_leg})"

    portfolio_return = pd.Series(dtype=float, name=portfolio_name)
    total_trade      = pd.Series(dtype=float, name=portfolio_name)

    # 수익률 윈저라이징
    monthly_rets_win = monthly_rets.clip(
        lower=monthly_rets.quantile(wins_threshold_temp),
        upper=monthly_rets.quantile(1 - wins_threshold_temp),
        axis = 1
    )

    prev_portfolio = None

    for i in range(len(month_ends)-1):
        start_date, end_date = month_ends[i], month_ends[i+1]

        # 시가총액 및 유동성 필터링
        if sort_mode in ['double', 'spread']:
            cap_series      = mkt_cap.loc[start_date]
            cap_quantile    = pd.qcut(cap_series, q=cap_cut, labels=False)
            filtered_index  = cap_series[cap_quantile == c_temp].index
            series          = trading_value_60.loc[start_date, filtered_index]
        else:
            series          = trading_value_60.loc[start_date]

        threshold       = series.quantile(trading_threshold_temp)
        filtered        = series[series > threshold]
        factor_filtered = factor_df.loc[start_date, filtered.index].dropna()

        # 종목 선정 (분위수 or 롱숏)
        if sort_mode == 'spread':
            low_th  = factor_filtered.quantile(0.3)
            high_th = factor_filtered.quantile(0.7)
            if spread_leg == 'long':
                basket = factor_filtered[factor_filtered <= low_th].index
            else:
                basket = factor_filtered[factor_filtered > high_th].index
        else:
            quantiles = pd.qcut(factor_filtered, q=q_cut, labels=False)
            basket    = factor_filtered[quantiles == n_temp].index

        # Amihud High Cost 종목 선정
        illiq_startdate = illiq.loc[start_date].dropna()
        threshold       = illiq_startdate.quantile(1-Amihud_threshold_temp)
        illiq_top       = illiq_startdate[illiq_startdate >= threshold].index

        # 비중 할당
        if prev_portfolio is None:
            prev_weights = pd.Series(0, index=basket)
        else:
            prev_weights = prev_portfolio / prev_portfolio.sum()

        if weight_method_temp == 'Equal':
            target_weights = pd.Series(1.0/len(basket), index=basket)
        else: # 'Cap'
            cap_seg        = mkt_cap.loc[start_date, basket]
            target_weights = cap_seg / cap_seg.sum()

        # 거래 비용 및 NAV 업데이트
        all_index = target_weights.index.union(prev_weights.index)
        target_w  = target_weights.reindex(all_index, fill_value=0)
        prev_w    = prev_weights.reindex(all_index, fill_value=0)
        delta_w   = target_w - prev_w

        trade_amounts = abs(delta_w) * NAV
        cost_rate     = np.where(delta_w.index.isin(illiq_top), high_cost, low_cost)
        trade_cost    = (trade_amounts * cost_rate).sum()

        NAV_new       = NAV - trade_cost

        current_portfolio_value = target_weights * NAV_new
        ret_seg                 = monthly_rets_win.loc[end_date, basket]
        next_portfolio_value    = current_portfolio_value * (ret_seg + 1)

        NAV_new                 = next_portfolio_value.sum()
        portfolio_ret           = NAV_new / NAV - 1

        prev_portfolio          = next_portfolio_value
        NAV                     = NAV_new

        total_trade.loc[start_date]    = trade_amounts.sum()
        portfolio_return.loc[end_date] = portfolio_ret

    # 결과 반환
    return {'return': portfolio_return, 'trade': total_trade}

In [3]:
def select_columns(df, fixed_options):
    # {기존 이름: 새 이름} 딕셔너리 생성
    rename_dict = {
        c: "_".join([c.split('_')[i] for i, v in fixed_options.items() if v == "*"])
        for c in df.columns if all(c.split('_')[i] == v or v == "*" for i, v in fixed_options.items())
    }
    
    # 컬럼 선택 후 반환
    return df[rename_dict.keys()].rename(columns=rename_dict)

---
##### **설정**

In [4]:
start_point       = '2013-04-30'                     # 백테스트 기간 설정
end_point         = '2026-02-28'

q_cut             = 5                                # 포트폴리오를 나누는 분위 수    e.g. 5 → 5분위 
n                 = [0, 1, 2, 3, 4]                  # 0 : 하위 ~ (q_cut - 1) : 상위  e.g. [0, 1, 2, 3, 4]

weight_method     = ['Equal', 'Cap']                 # ['Equal', Cap']
cost              = [(0.008, 0.003), (0, 0)]         # (high_cost, low_cost)          e.g. [(0.008, 0.003), (0, 0)]
wins_threshold    = [0.01, 0.00]                     # 수익률 윈저라이징              e.g. [0.01, 0.05]
trading_threshold = [0.1, 0.0]                       # 60일 평균 거래대금 필터링      e.g. [0.1, 0.0]
Amihud_threshold  = [0.2, 0.0]                       # 유동성 기준                    e.g. [0.2, 0.0]

initial_NAV       = 1                                # 초기값

##### **계산 실행**

In [5]:
# 일별 수익률과 거래대금을 이용해 Amihud illiquidity 계산
daily_ret        = adj_close.pct_change(fill_method=None)
daily_illiq      = (daily_ret.abs()/trading_value)
illiq            = daily_illiq.resample('ME').mean()

In [6]:
# monthly_rets        = total_adj_close.resample('QE').pct_change(fill_method=None) # 분기 리밸런싱
monthly_rets        = total_adj_close.resample('ME').last().pct_change(fill_method=None)

portfolio_df = pd.DataFrame()
trade_df     = pd.DataFrame()

month_ends  = pd.date_range(start=start_point, end=end_point, freq='ME')

In [7]:
# 모든 조합 만들기
param_list = [
    (wins_threshold_temp, weight_method_temp, cost_temp, trading_threshold_temp, Amihud_threshold_temp, n_temp)
    for wins_threshold_temp in wins_threshold
    for weight_method_temp in weight_method
    for cost_temp in cost
    for trading_threshold_temp in trading_threshold
    for Amihud_threshold_temp in Amihud_threshold
    for n_temp in n
]

# 병렬 실행
results = Parallel(n_jobs=-1)(
    delayed(run_strategy)(*params, monthly_rets, illiq) for params in tqdm(param_list)
)

# 결과 합치기
portfolio_df = pd.DataFrame({r['return'].name: r['return'] for r in results})
trade_df     = pd.DataFrame({r['trade'].name: r['trade'] for r in results})

100%|██████████| 160/160 [01:57<00:00,  1.36it/s]


##### **Output**

In [8]:
portfolio_df.tail()

,Equal_R0.01_H80L30_T0.1_A0.2_Q(1/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(2/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(3/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(4/5),Equal_R0.01_H80L30_T0.1_A0.2_Q(5/5),Equal_R0.01_H80L30_T0.1_A0.0_Q(1/5),Equal_R0.01_H80L30_T0.1_A0.0_Q(2/5),Equal_R0.01_H80L30_T0.1_A0.0_Q(3/5),Equal_R0.01_H80L30_T0.1_A0.0_Q(4/5),Equal_R0.01_H80L30_T0.1_A0.0_Q(5/5),...,Cap_R0.0_H0L0_T0.0_A0.2_Q(1/5),Cap_R0.0_H0L0_T0.0_A0.2_Q(2/5),Cap_R0.0_H0L0_T0.0_A0.2_Q(3/5),Cap_R0.0_H0L0_T0.0_A0.2_Q(4/5),Cap_R0.0_H0L0_T0.0_A0.2_Q(5/5),Cap_R0.0_H0L0_T0.0_A0.0_Q(1/5),Cap_R0.0_H0L0_T0.0_A0.0_Q(2/5),Cap_R0.0_H0L0_T0.0_A0.0_Q(3/5),Cap_R0.0_H0L0_T0.0_A0.0_Q(4/5),Cap_R0.0_H0L0_T0.0_A0.0_Q(5/5)
2025-10-31,-0.014536,-0.004173,0.000630,0.011189,0.038455,-0.014154,-0.003472,0.001421,0.012109,0.039087,...,0.017711,0.192667,0.269847,0.151045,0.138899,0.017711,0.192667,0.269847,0.151045,0.138899
2025-11-30,0.007771,-0.003299,-0.008054,-0.027716,-0.018981,0.007923,-0.003082,-0.007854,-0.027499,-0.018810,...,0.004238,-0.040499,-0.013338,-0.049342,-0.055861,0.004238,-0.040499,-0.013338,-0.049342,-0.055861
2025-12-31,0.035957,0.015142,0.014556,0.007928,0.025117,0.036349,0.015911,0.015466,0.008984,0.025706,...,0.149242,0.106744,0.001963,-0.006332,0.007717,0.149242,0.106744,0.001963,-0.006332,0.007717
2026-01-31,0.073694,0.094490,0.082582,0.108517,0.121339,0.073855,0.094644,0.082757,0.108692,0.121609,...,0.294231,0.282523,0.166392,0.185151,0.233945,0.294231,0.282523,0.166392,0.185151,0.233945
2026-02-28,0.042776,0.039743,0.055764,0.035818,0.034821,0.042962,0.039913,0.055923,0.035949,0.035093,...,0.290675,0.151316,0.169441,0.139414,0.045599,0.290675,0.151316,0.169441,0.139414,0.045599


---
## **1. 분위수 검증**

##### **포트폴리오 선택**

In [41]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

subset = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

In [42]:
subset.tail()

,Q(1/5),Q(2/5),Q(3/5),Q(4/5),Q(5/5)
2025-10-31,-0.014536,-0.004173,0.000630,0.011189,0.038455
2025-11-30,0.007771,-0.003299,-0.008054,-0.027716,-0.018981
2025-12-31,0.035957,0.015142,0.014556,0.007928,0.025117
2026-01-31,0.073694,0.094490,0.082582,0.108517,0.121339
2026-02-28,0.042776,0.039743,0.055764,0.035818,0.034821


In [43]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 데이터 준비
nav, log_cum = (subset + 1).cumprod(), np.log1p(subset).cumsum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Cumulative NAV", "Log Cumulative Return"])

# 단일 루프로 두 그래프 그리기
for col in subset.columns:
    fig.add_trace(go.Scatter(x=nav.index, y=nav[col], mode='lines', name=col, legendgroup=col), row=1, col=1)
    fig.add_trace(go.Scatter(x=log_cum.index, y=log_cum[col], mode='lines', name=col, legendgroup=col), row=1, col=2)

# 레이아웃 및 테두리 설정
fig.update_layout(title="Portfolio Performance Comparison", height=500, width=1000, template="plotly_white")
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 x축 테두리
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 y축 테두리

fig.show()

---
## **2. Double sort : Size Control**


##### **계산 실행**

In [12]:
cap_cut           = 5                          # 포트폴리오를 나누는 분위 수 e.g. 5 → 5분위 
c                 = [0, 1, 2, 3, 4]    

In [13]:
trading_threshold_temp = trading_threshold[0]
weight_method_temp     = weight_method[1]
cost_temp              = cost[0]
wins_threshold_temp    = wins_threshold[0]
trading_threshold_temp = trading_threshold[0]
Amihud_threshold_temp  = Amihud_threshold[0]

In [14]:
# --- 병렬 실행 ---
results_double = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        wins_threshold_temp, weight_method_temp, cost_temp, 
        trading_threshold_temp, Amihud_threshold_temp, n_temp,
        monthly_rets, illiq,    
        sort_mode='double',      # 'double'로 변경
        c_temp=c_temp            # 시가총액 분위수 추가
    )
    for c_temp in c
    for n_temp in n
)

# --- 결과 합치기 ---
cap_cut_portfolio_df = pd.DataFrame({r['return'].name: r['return'] for r in results_double})

##### **Output**

In [15]:
cap_cut_portfolio_df.tail()

,C(1/5)_Q(1/5),C(1/5)_Q(2/5),C(1/5)_Q(3/5),C(1/5)_Q(4/5),C(1/5)_Q(5/5),C(2/5)_Q(1/5),C(2/5)_Q(2/5),C(2/5)_Q(3/5),C(2/5)_Q(4/5),C(2/5)_Q(5/5),...,C(4/5)_Q(1/5),C(4/5)_Q(2/5),C(4/5)_Q(3/5),C(4/5)_Q(4/5),C(4/5)_Q(5/5),C(5/5)_Q(1/5),C(5/5)_Q(2/5),C(5/5)_Q(3/5),C(5/5)_Q(4/5),C(5/5)_Q(5/5)
2025-10-31,-0.006238,-0.032746,-0.020804,-0.029318,-0.049814,-0.040397,-0.024926,-0.037457,-0.021057,-0.024923,...,-0.005986,-0.013533,0.017637,0.011283,0.069631,0.184422,0.112794,0.281069,0.178507,0.146883
2025-11-30,-0.011713,-0.023235,-0.012134,-0.036435,-0.055698,0.004798,0.000599,-0.021938,-0.045016,-0.033390,...,0.006763,-0.011426,-0.010351,-0.016263,0.024455,-0.048257,0.025658,-0.031600,-0.071885,-0.063590
2025-12-31,0.021433,0.008323,0.007676,-0.008919,0.043120,0.039721,0.000676,0.014767,0.021280,0.000930,...,0.028555,0.001938,0.022831,-0.024994,0.072006,0.137658,0.108944,-0.007472,-0.012837,0.000337
2026-01-31,0.001812,0.014350,0.038234,0.068317,0.059496,0.024559,0.037355,0.033783,0.037925,0.070347,...,0.113859,0.113800,0.102273,0.122021,0.183901,0.286413,0.289826,0.139068,0.207102,0.226681
2026-02-28,0.018649,0.010824,0.004646,-0.017497,-0.021119,0.017408,0.027187,0.000952,0.037305,0.002306,...,0.063063,0.056746,0.054617,0.048888,0.080353,0.290782,0.139529,0.177102,0.136186,0.023403


---
##### **Plot**

##### **Code**

In [16]:
# 월별 수익률 시리즈 기준 CAGR 계산
def CAGR(series):
    s = series.dropna()
    if s.empty: return np.nan
    return (1 + s).prod() ** (365.25 / (s.index[-1] - s.index[0]).days) - 1

In [17]:
# 2. Heatmap 매트릭스 생성
cagrs = cap_cut_portfolio_df.apply(CAGR)
cagrs.index = cagrs.index.str.split('_', expand=True)
cagr_matrix = cagrs.unstack()

In [18]:
import plotly.express as px

# 3. 시각화
fig = px.imshow(
    cagr_matrix, text_auto=".1%", color_continuous_scale="Blues", aspect="equal",
    title="CAGR Heatmap by Cap vs Factor Quintile", width=600, height=600,
    labels=dict(x="Factor Quintile (Q)", y="Market Cap Quintile (C)", color="CAGR")
)

fig.show()

In [19]:
factors        = pd.read_csv("../../00_input/factors monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = cap_cut_portfolio_df['C(1/5)_Q(1/5)']
portfolio.name = 'Return'

In [23]:
import statsmodels.api as sm

# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀 (Newey-West 표준오차)
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})

# 5. 결과 출력
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.825
Model:                            OLS   Adj. R-squared:                  0.821
Method:                 Least Squares   F-statistic:                     188.1
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           3.64e-57
Time:                        00:00:30   Log-Likelihood:                 363.81
No. Observations:                 154   AIC:                            -717.6
Df Residuals:                     149   BIC:                            -702.4
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0085      0.002      5.131      0.0

In [24]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# 포트폴리오 설정
portfolio = cap_cut_portfolio_df['C(1/5)_Q(1/5)']

# 데이터 준비
nav = (1 + portfolio).cumprod()  # 누적 NAV
log_cum = np.log1p(portfolio).cumsum()  # 로그 누적 수익률

# 서브플롯 (가로 2개)
fig = make_subplots(
    rows=1, cols=2, 
    subplot_titles=["Cumulative NAV", "Log Cumulative Return"],
    horizontal_spacing=0.15
)

# 첫 번째 그래프: 누적 NAV
fig.add_trace(
    go.Scatter(
        x=nav.index, 
        y=nav.values, 
        mode='lines', 
        name='C(1/5)_Q(1/5)',
        line=dict(color='#1f77b4', width=2)
    ),
    row=1, col=1
)

# 두 번째 그래프: 로그 누적 수익률
fig.add_trace(
    go.Scatter(
        x=log_cum.index, 
        y=log_cum.values, 
        mode='lines', 
        name='C(1/5)_Q(1/5)',
        line=dict(color='#1f77b4', width=2),
        showlegend=False
    ),
    row=1, col=2
)

# 레이아웃 설정
fig.update_layout(
    title="Portfolio Performance: C(1/5)_Q(1/5)",
    height=500, 
    width=1000,
    template="plotly_white",
    hovermode='x unified'
)

# 축 레이블 설정
fig.update_xaxes(title_text="Date", row=1, col=1)
fig.update_xaxes(title_text="Date", row=1, col=2)
fig.update_yaxes(title_text="Cumulative NAV", row=1, col=1)
fig.update_yaxes(title_text="Log Cumulative Return", row=1, col=2)

fig.show()

In [25]:
import pandas as pd
from module import performance_metrics

# 1) 대상 포트폴리오 수익률 시리즈
ret = cap_cut_portfolio_df['C(1/5)_Q(1/5)'].dropna().copy()
ret.name = 'Return'

# 2) 거래대금 시리즈 (trade_df 이름이 다르면 해당 변수명으로 바꿔주세요)
if 'cap_cut_trade_df' in globals() and 'C(1/5)_Q(1/5)' in cap_cut_trade_df.columns:
    trade = cap_cut_trade_df['C(1/5)_Q(1/5)'].reindex(ret.index).fillna(0.0)
elif 'trade_df' in globals() and 'C(1/5)_Q(1/5)' in trade_df.columns:
    trade = trade_df['C(1/5)_Q(1/5)'].reindex(ret.index).fillna(0.0)
else:
    # turnover 계산은 0으로 나오지만, 나머지 성과지표는 계산 가능
    trade = pd.Series(0.0, index=ret.index, name='Trade')

# 3) module.py 형식에 맞는 portfolio DataFrame 구성
portfolio = pd.DataFrame(index=ret.index)
portfolio['Return'] = ret
portfolio['Trade'] = trade
portfolio['NAV'] = (1 + portfolio['Return']).cumprod()

# initial_NAV를 명시적으로 1로 맞추기 위해 시작점 한 줄 추가
start_row = pd.DataFrame(
    {'Return': [0.0], 'Trade': [0.0], 'NAV': [1.0]},
    index=[portfolio.index.min() - pd.offsets.MonthEnd(1)]
)
portfolio = pd.concat([start_row, portfolio]).sort_index()

# 4) 성과지표 계산
metrics = performance_metrics(portfolio)

# 보기 좋게 출력
metrics_df = pd.Series(metrics, name='C(1/5)_Q(1/5)').to_frame()
metrics_df.style.format("{:.4f}")

,C(1/5)_Q(1/5)
CAGR,0.1696
Volatility (ann.),0.1891
Sharpe Ratio,0.9281
MDD,-0.2245
Average Turnover (monthly),0.0000


---
## **3. Parameter Sensitivity**
`0. 포트폴리오 계산`에서 계산한 포트폴리오 `portfolio_df` 데이터 사용

---
#### **1) Weight sensitivity**

##### **포트폴리오 선택**

In [26]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_equal = select_columns(
    portfolio_df,
    {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

# 시총가중 포트폴리오
subset_cap = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "*"}
)

##### **Code**

In [27]:
# 데이터 준비
log_cum_eq, log_cum_cap = np.log1p(subset_equal).cumsum(), np.log1p(subset_cap).cumsum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Equal-weighted Portfolios", "Cap-weighted Portfolios"])

# 단일 루프로 두 그래프 동시에 그리기
for col in log_cum_eq.columns:
    # 1열: Equal 가중 그래프
    fig.add_trace(go.Scatter(x=log_cum_eq.index, y=log_cum_eq[col], mode='lines', name=f"Equal_{col}", legendgroup=col), row=1, col=1)
    # 2열: Cap 가중 그래프
    fig.add_trace(go.Scatter(x=log_cum_cap.index, y=log_cum_cap[col], mode='lines', name=f"Cap_{col}", legendgroup=col), row=1, col=2)

# 레이아웃 및 테두리 설정
fig.update_layout(title="Equal vs Cap Weighted Portfolios (Log Cumulative Returns)", height=500, width=1000, template="plotly_white")
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 x축 테두리
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True) # 각 서브플롯 y축 테두리

fig.show()

---
#### **2) Cost sensitivity**

##### **포트폴리오 선택**

In [28]:
# 기본 : {0: "Equal", 1: "R0.01", 2: "H80L30", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
# -----------------------------
# R: Return winsorization 
# H: High cost(bp)
# L: Low cost(bp)
# T: Trading value
# A: Amihud liquidity
# -----------------------------

# 동일가중 포트폴리오
subset_cost = select_columns(
    portfolio_df,
    {0: "Cap", 1: "R0.01", 2: "*", 3: "T0.1", 4:"A0.2", 5: "Q(1/5)"}
)

##### **Code**

In [29]:
# 데이터 준비
log_cum_cost = np.log1p(subset_cost).cumsum()

# DataFrame 렌더링 & 레이아웃 설정
fig = px.line(
    log_cum_cost, 
    title="Equal-weighted Portfolios (Cost Parameter Variation, Log Cumulative Return)",
    labels={"value": "Log Cumulative Return", "index": "Date", "variable": "Cost Parameter"},
    template="plotly_white", width=900, height=600
)

# 스타일 통일 적용
fig.update_xaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)
fig.update_yaxes(showline=True, linewidth=1, linecolor='gray', mirror=True)

fig.show()

---
## **4. 스프레드 회귀 (Factor regression)**

##### **데이터 로드**

In [30]:
factors        = pd.read_csv("../../00_input/factors monthly.csv", index_col=0, parse_dates=True)   # 팩터(MKT, SMB, HML, MOM, RF) (독립변수)
portfolio      = portfolio_df['Equal_R0.01_H80L30_T0.1_A0.2_Q(1/5)']
portfolio.name = 'Return'

##### **계산 실행**

In [31]:
cap_cut        = 2                  # 포트폴리오를 나누는 분위 수 e.g. 2 → 2분위 
c              = [0, 1]    
spread_legs    = ['long', 'short']

In [32]:
# 단일 병렬 실행
results_spread = Parallel(n_jobs=-1)(
    delayed(run_strategy)(
        wins_threshold[0], weight_method[0], cost[0], 
        trading_threshold[0], Amihud_threshold[0], None, # spread 모드에서는 n_temp가 무시되므로 None
        monthly_rets, illiq,                        
        sort_mode='spread', c_temp=c_temp, spread_leg=spread_leg
    )
    for c_temp in c
    for spread_leg in spread_legs
)

factor_portfolio_df = pd.DataFrame({r['return'].name: r['return'] for r in results_spread})

In [33]:
factor_portfolio_df.tail()

,C(1/2)_Q(long),C(1/2)_Q(short),C(2/2)_Q(long),C(2/2)_Q(short)
2025-10-31,-0.024135,-0.018316,0.006488,0.075113
2025-11-30,0.000860,-0.036231,-0.000978,-0.006743
2025-12-31,0.038115,0.017619,0.022126,0.025790
2026-01-31,0.022761,0.064210,0.134701,0.177150
2026-02-28,0.023079,0.003496,0.049023,0.071082


In [34]:
# long-short 스프레드 계산
small_ls = factor_portfolio_df["C(1/2)_Q(long)"] - factor_portfolio_df["C(1/2)_Q(short)"]
big_ls   = factor_portfolio_df["C(2/2)_Q(long)"] - factor_portfolio_df["C(2/2)_Q(short)"]

# 두 그룹 평균 → 최종 factor return
factor_return = (small_ls + big_ls) / 2
factor_return.name = "Factor"

##### **Output : ivol 팩터 포트폴리오** 

In [35]:
factor_return.tail()

2025-10-31   -0.037222
2025-11-30    0.021428
2025-12-31    0.008416
2026-01-31   -0.041949
2026-02-28   -0.001239
Name: Factor, dtype: float64

##### **회귀분석**

##### **Code**

In [36]:
# 1. 두 데이터 공통 구간 맞추기
df = pd.concat([portfolio, factors], axis=1, join="inner").dropna()

In [37]:
import statsmodels.api as sm

# 2. 종속변수: 포트폴리오 초과수익률
y = df['Return'] - df['RF']

# 3. 독립변수: MKT, SMB, HML, MOM
X = pd.DataFrame({
    "MKT": df['KOSPI'] - df['RF'],
    "SMB": df['SMB'],
    "HML": df['HML'],
    "MOM": df['MOM']
}, index=df.index)

X = sm.add_constant(X, has_constant='add')

# 4. OLS 회귀
model = sm.OLS(y, X).fit(cov_type="HAC", cov_kwds={"maxlags":12})  # Newey-West 표준오차

##### **기존 결과**

In [38]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.891
Model:                            OLS   Adj. R-squared:                  0.889
Method:                 Least Squares   F-statistic:                     225.1
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           4.45e-62
Time:                        00:00:37   Log-Likelihood:                 415.82
No. Observations:                 154   AIC:                            -821.6
Df Residuals:                     149   BIC:                            -806.5
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const          0.0006      0.001      0.491      0.6

---

##### **Code**

In [39]:
# factor_return 추가
df["Factor"] = factor_return.loc[df.index]

# 독립변수: MKT, SMB, HML, MOM, MyFactor
X_ext = X.copy()
X_ext["Factor"] = df["Factor"]

X2 = sm.add_constant(X_ext)

# 4. OLS 회귀
model = sm.OLS(y, X2).fit(cov_type="HAC", cov_kwds={"maxlags":12})

##### **새로운 팩터를 포함한 결과**

In [40]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.926
Model:                            OLS   Adj. R-squared:                  0.923
Method:                 Least Squares   F-statistic:                     410.1
Date:                Tue, 24 Mar 2026   Prob (F-statistic):           8.44e-85
Time:                        00:00:38   Log-Likelihood:                 444.91
No. Observations:                 154   AIC:                            -877.8
Df Residuals:                     148   BIC:                            -859.6
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.0017      0.001     -2.194      0.0

---
## **5. LongShort Test**